# 05 · Build candidate tables
Train: reliable trips only, every 5th GPS point (1 Hz points are highly correlated), points whose aligned reference arc
is > 50 m away are dropped. Student-test: every point of every trip.

In [1]:
import sys, os
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "pathsnap").is_dir()), None)
if _root is None:
    raise RuntimeError("src/pathsnap not found: unzip all zips of the set into the same pathsnap/ folder")
sys.path.insert(0, str(_root / "src")); os.chdir(_root / "notebooks")
import numpy as np, pandas as pd
from pathsnap import io, labels, features

os.makedirs("../outputs/data", exist_ok=True)
lq = pd.read_csv("../outputs/label_quality.csv", dtype={"trip": str}).set_index("trip")

In [2]:
def build(split, step, filter_labels):
    out = []
    for tid in io.list_trips(split):
        if filter_labels and not lq.loc[tid, "reliable"]:
            continue
        tr = io.load_trip(split, tid)
        lab, ld = labels.align_route(tr)
        df, _ = features.candidate_table(tr, lab, step=step)
        df["label_dist"] = ld[df.point.values]
        if filter_labels:
            df = df[df.label_dist <= 50]
        out.append(df)
    return pd.concat(out, ignore_index=True)

train = build("train", 5, True)
test = build("student_test", 1, False)
train.to_parquet("../outputs/data/train.parquet")
test.to_parquet("../outputs/data/student_test.parquet")

In [3]:
for n, d in [("train", train), ("student_test", test)]:
    print(n, d.shape, "trips", d.trip.nunique(), "points", d.groupby(["trip", "point"]).ngroups,
          "pos rate", round(d.y.mean(), 3), "points w/o true arc", round(1 - d.groupby(["trip", "point"]).y.max().mean(), 4))

train (245706, 23) trips 78 points 36178 pos rate 0.147 points w/o true arc 0.0048
student_test (130963, 23) trips 10 points 19160 pos rate 0.146 points w/o true arc 0.0037
